# 06 · Linaje, SQL Athena y contratos de extracción

> **Estado de evidencia:** los CSV suministrados son plantillas sin observaciones. Las celdas calculan resultados únicamente si se cargan extractos reales. No se fabrican hallazgos. Las fórmulas constituyen especificaciones metodológicas sujetas a aprobación del contrato de métricas.

**Convenciones:** $i$ crédito, $c$ cohorte de originación, $t$ corte calendario, $m$ meses desde originación (MOB), $D_{i,t}$ días de mora, $B_{i,t}$ saldo capital, $L_{i,t}$ cupo. Los límites de ventanas y las reglas de castigo deben parametrizarse.

## 1. Objetivo, alcance y decisiones

Se distinguen observaciones a nivel crédito-corte, crédito-MOB, evento y cliente. Se prohíben joins uno-a-muchos sin agregación previa. La trazabilidad debe conservar fecha de vigencia y disponibilidad.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE = Path.cwd()
if not (BASE/'plantilla_panel_mensual.csv').exists():
    BASE = Path('/mnt/data/estudio_riesgo_credito')
assert BASE.exists(), 'Ubique el notebook junto a las plantillas CSV'
plt.rcParams.update({'figure.figsize':(9,4.8),'axes.grid':True,'grid.alpha':.2})
def load(name, parse_dates=None):
    path=BASE/name
    df=pd.read_csv(path,parse_dates=parse_dates or [])
    print(f'{name}: {len(df):,} registros, {len(df.columns)} columnas')
    return df

## Inventario y descubrimiento de esquemas

La disponibilidad de una variable requiere demostrar: tabla física, columna, llave, fecha de evento, fecha de publicación, unidad, calidad y permiso de uso. Una coincidencia por nombre **no demuestra** que el campo tenga la semántica requerida.

In [ ]:
sources=load('inventario_fuentes.csv');dictionary=load('diccionario_variables.csv')
display(sources.head(30));display(dictionary.head(15))

## SQL de descubrimiento seguro

Consultar metadatos antes de escanear hechos. Filtrar particiones y fechas para controlar costo. **SQL propuesto, no ejecutado contra Athena:**

In [ ]:
sql="""SELECT table_schema, table_name, column_name, data_type
FROM information_schema.columns
WHERE regexp_like(lower(column_name), 'merchant|client|city|municip|transaction|score|amount|date')
ORDER BY table_schema, table_name, ordinal_position"""
print(sql)

## Especificación de snapshot y join temporal

Para registro de saldo $b_{i,s}$ y corte $t$, seleccionar $s^*=\max\{s:s\le t\}$ **solo si** la tabla es un histórico de estados válidos y la política admite arrastre. En una tabla de hechos de cierre, exigir $s=t$ y marcar ausencias. Para vigencias tipo SCD2:
$$a_{j}\le t<b_j,\quad \sum_j\mathbf1(a_j\le t<b_j)\le1.$$
Verificar solapamientos antes de integrar. Los snapshots de transacciones requieren preagregación por comercio y ventana.

In [ ]:
sql="""-- Plantilla lógica: adaptar nombres y tipos confirmados en Athena
SELECT a.loan_key, a.merchant_key, a.cutoff_date_key,
       a.davs_past_due AS dpd, b.principal_amount AS saldo_capital
FROM awsdatacatalog.bold_gold_loan.mart.fact_arrears_history a
LEFT JOIN awsdatacatalog.bold_gold_loan.fact_loan_balance b
 ON a.loan_key=b.loan_key
AND a.merchant_key=b.merchant_key
AND a.cutoff_date_key=b.cutoff_date_key
-- Agregar filtro de partición y control de unicidad antes de producción
"""
print(sql)

## Contrato de campos y controles de entrega

Para cada campo $x$, registrar $\{nombre,tipo,unidad,granularidad,fecha\_disponibilidad,origen,nullable,regla\}$. Aceptación del extracto: unicidad por llave, fechas coherentes, DPD no negativo (salvo convenciones documentadas), reconciliación de saldos, integridad de llaves y cobertura por cohorte. Los nombres en las plantillas son **objetivos de extracción**, no columnas verificadas en AWS.

In [ ]:
metrics=load('contratos_metricas.csv')
display(metrics)
print('IMPORTANTE: revisar discrepancias SQL original: davs_past_due vs days_past_due, vigencia_desde, >30 vs >=30, joins migrados y granularidad de versiones.')

## Limitaciones y siguiente ejecución

**Sin datos reales** no se pueden inferir tasas, significancia, calidad predictiva ni relaciones causales. Cargar las plantillas desde Athena con llaves y fechas; verificar los resultados de los controles antes de interpretar gráficos. Para producción: pruebas temporales, monitoreo de sesgo de selección, documentación de fuentes y revisión de privacidad.